# 025-a-contrario-gate

# Imports

In [ ]:
import os

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from matplotlib.figure import Figure
from tqdm.auto import tqdm

from uavloc.data.drone_frames import DroneFrame, frame_place, load_frames
from uavloc.data.match_pairs import village_split
from uavloc.data.multidate import load_date
from uavloc.datasets import dataset_ref
from uavloc.features.factory import correlator_config
from uavloc.features.segmentation import SegmentationFeatures, load_segmenter
from uavloc.matching.acontrario import (NullModel, OwnMapNull, VillageNull, at_epsilon, median_ms_per_nfa_correction,
                                        score_nfa_draws)
from uavloc.matching.benchmark import (HARM_MARGIN_M, Draw, benchmark_draws, error_stats, median_ms_per_correction,
                                       score_draws, summarise)
from uavloc.matching.correlator import CorrelatorConfig, PositionCorrelator
from uavloc.matching.tuning import TuningGrid, tuned_config
from uavloc.track import REPO_ROOT, experiment_dir, log_run, set_seed

# Params

In [ ]:
EXP = "025-a-contrario-gate"
NOTE = ""
LOG_RUN = True

ARM = "static"                    # "static" (025.1, = 022.2) | "nfa-villages" (025.2) | "nfa-own-map" (025.3)
GATE = "accept-reject"            # static arm: 022.2's gate, LOPO-tuned
TEMPERATURE_GRID = [0.25, 0.5, 1.0, 2.0]
S_MAX_GRID_M = [2.0, 3.0, 5.0]
C_MIN_GRID = [0.01, 0.03, 0.05]
TUNING_NOISE_M = 10.0
STATIC_MIN_APPLIED_SHARE = 0.5
NFA_EPSILON = 1.0                 # pre-registered; the curve below is a diagnostic, never chosen from
EPSILON_CURVE = [0.01, 0.1, 1.0, 10.0]
NULL_WINDOWS = 200
NULL_SEED = 42
OWN_MAP_EXCLUSION_M = 60.0

PLACES = ["bushcha-50p2422-25p8305", "birky-49p9105-23p9192"]
FRAMES_DATASET = "village-crossdate-frames"
HOLDOUT_DATASET = "village-crossdate-holdout"
EXCLUDE_PLACES = ["birky-49p9105-23p9192", "bushcha-50p2422-25p8305", "dubrova-49p5175-24p0765"]
N_VAL_VILLAGES = 10
SPLIT_SEED = 42
MAP_DATE = "newest"
MATCHING = "weighted-ncc"
CLASS_WEIGHTS = (0.0, 1.0, 1.0, 0.5, 0.0, 0.5)   # background, building, track, woody, field, water
NOISE_M = [5.0, 10.0, 15.0]
PRIMARY_NOISE_M = 10.0
N_DIRECTIONS = 8
SEED = 42
DEVICE = "mps"
TORCH_THREADS = 2                 # shared CPU
SEARCH_SCALE = 1.7
FOV_W_DEG = 60.0
SAT_FEATURE_CROP_PCT = 0.03
DRONE_FEATURE_CROP_PCT = 0.0
SEG_ARCHITECTURE = "lraspp"
SEG_CHECKPOINT = "pocs/mobilenet-lraspp-speed/model-lraspp-bn-satellite.pt"
SEG_CHANNELS = "all"
MATCH_GSD_M = 0.385

TIMING_DEVICE = "cpu"
TIMING_FRAMES = 8
TIMING_REPEATS = 3
NFA_PLOT_FLOOR = 1e-3

# Data

In [ ]:
set_seed(SEED)
torch.set_num_threads(TORCH_THREADS)

In [ ]:
NFA_ARM = ARM != "static"
train_villages, val_villages = village_split(EXCLUDE_PLACES, N_VAL_VILLAGES, SPLIT_SEED)
len(train_villages), val_villages

In [ ]:
RUN_CONFIG = {
    "method": "segmentation", "matching": MATCHING, "class_weights": CLASS_WEIGHTS,
    "noise_m": NOISE_M, "primary_noise_m": PRIMARY_NOISE_M, "n_directions": N_DIRECTIONS,
    "window": "rect", "search_scale": SEARCH_SCALE, "search_radius_m": None, "prior_sigma_m": None,
    "arm": ARM, "gate": GATE if ARM == "static" else "nfa",
    "tuning": {"temperatures": TEMPERATURE_GRID, "s_max_m": S_MAX_GRID_M, "c_min": C_MIN_GRID, "noise_m": TUNING_NOISE_M,
               "min_applied_share": STATIC_MIN_APPLIED_SHARE, "scheme": "leave-one-place-out",
               "temperature_by": "mean-error", "holdout": "both primary places pooled"} if ARM == "static" else None,
    "nfa": {"epsilon": NFA_EPSILON, "epsilon_curve": EPSILON_CURVE, "null_windows": NULL_WINDOWS, "null_seed": NULL_SEED,
            "null": "train-villages" if ARM == "nfa-villages" else "own-map",
            "own_map_exclusion_m": OWN_MAP_EXCLUSION_M if ARM == "nfa-own-map" else None,
            "raw_score": "weighted class NCC before z-score", "n_tests": "finite cells",
            "pose": "uniform position and heading"} if NFA_ARM else None,
    "fov_w_deg": FOV_W_DEG, "sat_feature_crop_pct": SAT_FEATURE_CROP_PCT, "drone_feature_crop_pct": DRONE_FEATURE_CROP_PCT,
    "seg_architecture": SEG_ARCHITECTURE, "seg_checkpoint_sat": SEG_CHECKPOINT, "seg_checkpoint_drone": SEG_CHECKPOINT,
    "seg_channels": SEG_CHANNELS, "drone_seg_gsd_m": MATCH_GSD_M, "match_gsd_m": MATCH_GSD_M,
    "frames_dataset": FRAMES_DATASET, "frames_dataset_ref": dataset_ref(FRAMES_DATASET),
    "holdout_dataset": HOLDOUT_DATASET, "holdout_dataset_ref": dataset_ref(HOLDOUT_DATASET),
    "places": PLACES, "map_date": MAP_DATE,
    "split": {"exclude": EXCLUDE_PLACES, "n_val": N_VAL_VILLAGES, "seed": SPLIT_SEED, "val": val_villages},
    "torch_threads": TORCH_THREADS,
}

In [ ]:
def load_bench(dataset: str, places: list[str]) -> dict[str, list[DroneFrame]]:
    """Frames of `dataset` grouped by place; raises if a place has none."""
    frames = load_frames(dataset)
    bench = {place: [f for f in frames if frame_place(f) == place] for place in places}
    missing = [place for place, members in bench.items() if not members]
    if missing:
        raise ValueError(f"{dataset} has no frames for {missing}")
    return bench

In [ ]:
benches = {"": load_bench(FRAMES_DATASET, PLACES), "holdout_": load_bench(HOLDOUT_DATASET, val_villages)}
{prefix or "primary": {place: len(m) for place, m in bench.items()} for prefix, bench in benches.items()}

# Model

In [ ]:
model = load_segmenter(REPO_ROOT / SEG_CHECKPOINT, DEVICE, SEG_ARCHITECTURE)
base_config = correlator_config({**RUN_CONFIG, "gate": "none"})
base_config

In [ ]:
def map_features(place: str, device: str, segmenter: torch.nn.Module) -> SegmentationFeatures:
    """`place`'s newest map, segmented once as the bench segments it."""
    return SegmentationFeatures(segmenter, segmenter, load_date(place, MAP_DATE), SEG_CHANNELS, MATCH_GSD_M,
                                MATCH_GSD_M, device)

In [ ]:
def make_correlator(place: str, device: str, segmenter: torch.nn.Module) -> PositionCorrelator:
    return PositionCorrelator(map_features(place, device, segmenter), base_config)

In [ ]:
geometry = PositionCorrelator(None, base_config)   # patch sizes only: they depend on the config and altitude
patch_sizes_m = {geometry.patch_size_m(f) for bench in benches.values() for m in bench.values() for f in m}
assert len(patch_sizes_m) == 1, patch_sizes_m   # one window size, so one village null serves every frame
patch_sizes_m

In [ ]:
def build_null() -> NullModel | None:
    """025.2: windows on the 023 train villages; 025.3: windows on the test place's own map."""
    if ARM == "nfa-villages":
        return VillageNull.build(train_villages, lambda place: map_features(place, DEVICE, model),
                                 next(iter(patch_sizes_m)), SAT_FEATURE_CROP_PCT, NULL_WINDOWS, NULL_SEED)
    if ARM == "nfa-own-map":
        return OwnMapNull(NULL_WINDOWS, OWN_MAP_EXCLUSION_M, NULL_SEED)
    return None

In [ ]:
null = build_null()

# Run

In [ ]:
def place_draws(place: str, members: list[DroneFrame]) -> tuple[list[Draw], pd.DataFrame | None]:
    """022.2's draws of one place (same frames, levels, directions, seed); NFA-scored at once for an NFA arm."""
    correlator = make_correlator(place, DEVICE, model)
    draws = [d for noise_m in NOISE_M for d in benchmark_draws(correlator, members, noise_m, N_DIRECTIONS, SEED)]
    if not NFA_ARM:
        return draws, None
    records = score_nfa_draws(draws, correlator, null, NFA_EPSILON)
    null.clear_cache()
    return draws, records

In [ ]:
scored = {(prefix, place): place_draws(place, members)
          for prefix, bench in benches.items() for place, members in tqdm(bench.items(), desc=prefix or "primary")}

In [ ]:
def other_place(held_out: str) -> str:
    """Leave-one-place-out: a held-out place is scored with values fitted on the other one."""
    return next(place for place in PLACES if place != held_out)

In [ ]:
grid = TuningGrid(tuple(TEMPERATURE_GRID), tuple(S_MAX_GRID_M), tuple(C_MIN_GRID), TUNING_NOISE_M,
                  STATIC_MIN_APPLIED_SHARE)
primary_draws = {place: scored[("", place)][0] for place in PLACES}

In [ ]:
static_folds = {place: tuned_config(primary_draws[other_place(place)], base_config, GATE, grid)
                for place in PLACES} if ARM == "static" else {}
holdout_fold = tuned_config([d for place in PLACES for d in primary_draws[place]], base_config, GATE, grid) \
    if ARM == "static" else None
{place: (c.temperature, c.gate_s_max_m, c.gate_c_min) for place, (c, _) in static_folds.items()}

In [ ]:
def static_config(prefix: str, place: str) -> CorrelatorConfig:
    """The primary bench scores each place with the other place's values; the holdout with values from both."""
    return static_folds[place][0] if prefix == "" else holdout_fold[0]

In [ ]:
def place_records(prefix: str, place: str) -> pd.DataFrame:
    draws, nfa_records = scored[(prefix, place)]
    records = nfa_records if NFA_ARM else score_draws(draws, static_config(prefix, place))
    records = records.assign(place=place)
    records["kept_prior"] = records["decision"] == "reject"
    records["harm"] = records["error_m"] > records["prior_error_m"] + HARM_MARGIN_M
    return records

In [ ]:
records = {prefix: pd.concat([place_records(prefix, place) for place in bench], ignore_index=True)
           for prefix, bench in benches.items()}
records[""].groupby(["place", "noise_m"])[["error_m", "harm", "kept_prior", "reachable"]].mean().round(3)

In [ ]:
def argmax_outcome(records: pd.DataFrame) -> pd.Series:
    """What the ungated argmax would have done: harmful (worse than the prior by the harm margin), helpful, neutral."""
    error = records["argmax_error_m"]
    return pd.Series(np.select([error > records["prior_error_m"] + HARM_MARGIN_M, error < records["prior_error_m"]],
                               ["harmful", "helpful"], "neutral"), index=records.index)

In [ ]:
def nfa_metrics(primary: pd.DataFrame) -> dict[str, float]:
    """The epsilon curve (diagnostic only) and the NFA of draws the argmax would harm vs help."""
    metrics = {}
    for epsilon in EPSILON_CURVE:
        stats = error_stats(at_epsilon(primary, epsilon))
        for key in ("mean_position_error_m", "harm_share", "accept_share"):
            metrics[f"eps_{epsilon:g}_{key}"] = stats[key]
    outcome = argmax_outcome(primary)
    for name in ("harmful", "helpful"):
        metrics[f"nfa_median_{name}"] = float(primary.loc[outcome == name, "nfa"].median())
        metrics[f"accept_share_of_{name}"] = float((primary.loc[outcome == name, "decision"] == "accept").mean())
    return metrics

In [ ]:
def run_metrics(records: pd.DataFrame, bench: dict[str, list[DroneFrame]], prefix: str) -> dict[str, float]:
    """022's metrics, plus max error, per-place accept and harm, and for an NFA arm the epsilon curve."""
    subsets = {"all": {f.name for m in bench.values() for f in m},
               **{place.split("-")[0]: {f.name for f in m} for place, m in bench.items()}}
    metrics = summarise(records, subsets, PRIMARY_NOISE_M)
    for noise_m, group in records.groupby("noise_m"):
        metrics[f"kept_prior_share_at_{noise_m:g}m"] = float(group["kept_prior"].mean())
        metrics[f"max_position_error_m_at_{noise_m:g}m"] = float(group["error_m"].max())
    primary = records[records["noise_m"] == PRIMARY_NOISE_M]
    metrics["kept_prior_share"] = metrics[f"kept_prior_share_at_{PRIMARY_NOISE_M:g}m"]
    metrics["max_position_error_m"] = metrics[f"max_position_error_m_at_{PRIMARY_NOISE_M:g}m"]
    for place, group in primary.groupby("place"):
        short = place.split("-")[0]
        metrics[f"accept_share_{short}"] = float((group["decision"] == "accept").mean())
        metrics[f"harm_share_{short}"] = float(group["harm"].mean())
    if NFA_ARM:
        metrics.update(nfa_metrics(primary))
    return {f"{prefix}{key}": value for key, value in metrics.items()}

In [ ]:
metrics = {key: value for prefix, bench in benches.items()
           for key, value in run_metrics(records[prefix], bench, prefix).items()}
{k: round(v, 5) for k, v in metrics.items() if "_at_" not in k and not k.startswith("holdout_")}

In [ ]:
{k: round(v, 5) for k, v in metrics.items() if "_at_" not in k and k.startswith("holdout_")}

In [ ]:
def timing_ms() -> float:
    """Median ms per correction on the CPU, drone inference included; an NFA arm adds the raw surface and its null."""
    segmenter = load_segmenter(REPO_ROOT / SEG_CHECKPOINT, TIMING_DEVICE, SEG_ARCHITECTURE)
    correlator = make_correlator(PLACES[0], TIMING_DEVICE, segmenter)
    frames = benches[""][PLACES[0]][:TIMING_FRAMES]
    if not NFA_ARM:
        return median_ms_per_correction(correlator, frames, PRIMARY_NOISE_M, N_DIRECTIONS, TIMING_REPEATS)
    return median_ms_per_nfa_correction(correlator, null, frames, PRIMARY_NOISE_M, N_DIRECTIONS, TIMING_REPEATS,
                                        NFA_EPSILON)

In [ ]:
metrics["ms_per_correction"] = timing_ms()
metrics["ms_per_correction"]

# Plots

In [ ]:
def plot_error_cdf(records: pd.DataFrame, title: str) -> Figure:
    """Error CDF per injected level, the level marked as a dotted line."""
    fig, ax = plt.subplots(figsize=(6, 4))
    for noise_m, group in records.groupby("noise_m"):
        errors = np.sort(group["error_m"].to_numpy())
        line, = ax.plot(errors, np.linspace(0, 1, len(errors)), label=f"{noise_m:g} m injected")
        ax.axvline(noise_m, color=line.get_color(), ls=":", lw=1)
    ax.set_xlabel("position error after correction, m"); ax.set_ylabel("share of draws")
    ax.set_title(title); ax.legend(); ax.grid(alpha=.3)
    fig.tight_layout()
    return fig

In [ ]:
def plot_nfa_by_outcome(records: pd.DataFrame, title: str) -> Figure:
    """log10 NFA at the primary level, split by what the ungated argmax would have done; epsilon dashed."""
    primary = records[(records["noise_m"] == PRIMARY_NOISE_M) & (records["status"] == "ok")]
    outcome = argmax_outcome(primary)
    log_nfa = np.log10(np.maximum(primary["nfa"], NFA_PLOT_FLOOR))
    bins = np.linspace(log_nfa.min(), log_nfa.max(), 30)
    fig, ax = plt.subplots(figsize=(6, 4))
    for name in ("helpful", "neutral", "harmful"):
        ax.hist(log_nfa[outcome == name], bins=bins, alpha=.5, label=f"{name} ({(outcome == name).sum()})")
    ax.axvline(np.log10(NFA_EPSILON), color="k", ls="--", lw=1)
    ax.set_xlabel(f"log10 NFA (floored at {NFA_PLOT_FLOOR:g})"); ax.set_ylabel("draws")
    ax.set_title(title); ax.legend(); ax.grid(alpha=.3)
    fig.tight_layout()
    return fig

In [ ]:
def plot_epsilon_curve(records: pd.DataFrame, title: str) -> Figure:
    """Accept and harm share at the primary level against epsilon (diagnostic; the run is epsilon = 1)."""
    primary = records[records["noise_m"] == PRIMARY_NOISE_M]
    stats = pd.DataFrame([error_stats(at_epsilon(primary, e)) for e in EPSILON_CURVE], index=EPSILON_CURVE)
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(stats.index, stats["accept_share"], "o-", label="accept share")
    ax.plot(stats.index, stats["harm_share"], "s-", label="harm share")
    ax.axvline(NFA_EPSILON, color="k", ls="--", lw=1)
    ax.set_xscale("log"); ax.set_xlabel("epsilon"); ax.set_ylabel("share of draws")
    ax.set_title(title); ax.legend(); ax.grid(alpha=.3)
    fig.tight_layout()
    return fig

In [ ]:
figures = [plot_error_cdf(r, f"{ARM}, {prefix or 'primary'} bench") for prefix, r in records.items()]
if NFA_ARM:
    figures += [plot(r, f"{ARM}, {prefix or 'primary'} bench") for prefix, r in records.items()
                for plot in (plot_nfa_by_outcome, plot_epsilon_curve)]

# Log

In [ ]:
for field in ("temperature", "gate_s_max_m", "gate_c_min"):
    RUN_CONFIG[field] = {place: getattr(c, field) for place, (c, _) in static_folds.items()} or None
    RUN_CONFIG[f"holdout_{field}"] = getattr(holdout_fold[0], field) if holdout_fold else None

In [ ]:
artifacts = experiment_dir(EXP) / "artifacts"
artifacts.mkdir(exist_ok=True)
run_id = log_run(exp=EXP, config=RUN_CONFIG, metrics=metrics, note=NOTE, figures=figures,
                 seed=SEED, device=DEVICE)["run"] if LOG_RUN else "dry"
for prefix, r in records.items():
    r.to_csv(artifacts / f"{prefix}records-{run_id}.csv", index=False)
if static_folds:
    tables = [t.assign(held_out=place) for place, (_, t) in static_folds.items()] + [holdout_fold[1].assign(held_out="holdout")]
    pd.concat(tables, ignore_index=True).to_csv(artifacts / f"tuning-{run_id}.csv", index=False)
if isinstance(null, VillageNull):
    pd.DataFrame([{"place": p.place, "lat": p.center.lat, "lon": p.center.lon, "yaw_deg": p.yaw_deg} for p in null.poses]
                 ).to_csv(artifacts / f"null-poses-{run_id}.csv", index=False)
run_id